# Single-PCA Encoding Drift — SLURM Orchestrator

Tests whether Poisson GLM encoding models trained on one day generalise to future days,
using a **shared global PCA basis** so that weight vectors are comparable across days.

**Phase 0** (one SLURM job per patient):  
Fits a single PCA preprocessing bundle on ALL words from ALL recording days.
Saves the bundle for use in Phases 1 and 2.

**Phase 1** (one SLURM job per (patient, train_date)):  
Fit a full-day GLM using the global bundle and alpha hyperparameters from the
existing nested-CV per-day results. Saves model weights and training word indices.

**Phase 2** (one SLURM job per (patient, train_date)):  
For each subsequent test_date, apply the Phase-1 model (with global bundle) to
test-day words and run N permutation tests (shuffle training X → apply global
bundle → refit GLM → evaluate).

**Why this matters**: the global bundle ensures that W vectors from different
days live in the same feature space, making cosine-distance drift analysis valid.

In [2]:
from pathlib import Path
import subprocess

import dill as pickle
import numpy as np
import pandas as pd

In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PROJ_DIR  = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT  = PROJ_DIR / 'vad_new'

WORKER_BUNDLE = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                     '/functional_drift/encoding_singlepca_global_bundle_worker.py')
WORKER_TRAIN  = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                     '/functional_drift/encoding_singlepca_train_worker.py')
WORKER_TEST   = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                     '/functional_drift/encoding_singlepca_test_worker.py')
PYTHON_BIN    = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS       = None   # None => scan all Y* folders
FORCE_RESUBMIT = False

# ── Speech type ────────────────────────────────────────────────────────────────
SPEECH_TYPE = 'filtered'   # 'filtered' or 'patient'

SOURCE_RUN = {
    'filtered': 'word_level_duration_cv_filtered_speech_per_day',
    'patient':  'word_level_duration_cv_patient_speech_per_day',
}[SPEECH_TYPE]

DRIFT_SUBDIR = f'encoding_singlepca_{SPEECH_TYPE}'

# ── Model params (must match the source run) ──────────────────────────────────
SPIKE_OFFSET_IDX = 0
GPT2_LAYER       = -1
N_PCA            = 100

# ── Permutation testing ────────────────────────────────────────────────────────
N_PERMUTATIONS = 50
N_JOBS_PERM    = 4

# ── SLURM — Phase 0 (CPU, fits global PCA) ────────────────────────────────────
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'

P0_PARTITION = 'Universal'
P0_GRES      = ''
P0_CPUS      = 8
P0_MEM       = '64G'
P0_TIME      = '04:00:00'

# ── SLURM — Phase 1 (GPU, fits full-day GLM) ──────────────────────────────────
P1_PARTITION = 'Universal'
P1_GRES      = 'gpu:1'
P1_CPUS      = 8
P1_MEM       = '64G'
P1_TIME      = '08:00:00'

# ── SLURM — Phase 2 (CPU, permutation testing) ────────────────────────────────
P2_PARTITION = 'Universal'
P2_GRES      = ''
P2_CPUS      = N_JOBS_PERM * 2
P2_MEM       = '32G'
P2_TIME      = '24:00:00'

# ── Logging ───────────────────────────────────────────────────────────────────
LOGS_DIR    = VAD_ROOT / f'encoding_singlepca_{SPEECH_TYPE}_slurm_logs'
SCRIPTS_DIR = VAD_ROOT / f'encoding_singlepca_{SPEECH_TYPE}_slurm_scripts'
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:   ', VAD_ROOT)
print('source run: ', SOURCE_RUN)
print('drift dir:  ', DRIFT_SUBDIR)
print(f'permutations: {N_PERMUTATIONS} x n_jobs={N_JOBS_PERM}')

vad root:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
source run:  word_level_duration_cv_filtered_speech_per_day
drift dir:   encoding_singlepca_filtered
permutations: 50 x n_jobs=4


## 1. Discover Patients and Recording Days

In [4]:
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def resolve_patient_inputs(patient):
    r = VAD_ROOT / patient
    emb = first_existing([r / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
                           r / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy'])
    cnt = first_existing([r / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
                           r / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy'])
    dur = first_existing([r / 'neural_embeddings' / 'word_durs.npy',
                           r / 'all_convo_recording' / 'word_durs.npy'])
    return emb, cnt, dur


def get_model_days(patient):
    """Days with completed nested-CV results (have pkl + tar + word_idx)."""
    base = VAD_ROOT / patient / 'encoding' / SOURCE_RUN
    if not base.exists():
        return []
    days = []
    for d in sorted(base.iterdir()):
        if (d.is_dir()
                and (d / f'{patient}_encoding_results_cv.pkl').exists()
                and (d / f'{patient}_encoding_models_cv.tar').exists()):
            idx_files = list(d.glob(f'{patient}_*_word_idx.npy'))
            if idx_files:
                days.append((d.name, d / f'{patient}_encoding_results_cv.pkl', idx_files[0]))
    return days


def get_test_days(patient):
    base = VAD_ROOT / patient / 'encoding' / SOURCE_RUN
    if not base.exists():
        return []
    return [d.name for d in sorted(base.iterdir())
            if d.is_dir() and list(d.glob(f'{patient}_*_word_idx.npy'))]


if PATIENTS is None:
    all_patients = sorted(p.name for p in VAD_ROOT.iterdir()
                          if p.is_dir() and p.name.startswith('Y'))
else:
    all_patients = list(PATIENTS)

overview = []
for pt in all_patients:
    mdays = get_model_days(pt)
    tdays = get_test_days(pt)
    n_pairs = sum(1 for (d, _, _) in mdays for t in tdays if t > d)
    overview.append(dict(patient=pt, n_model_days=len(mdays),
                         n_test_days=len(tdays), n_pairs=n_pairs))

overview_df = pd.DataFrame(overview)
print(f'Patients with model days: {int((overview_df.n_model_days > 0).sum())}')
print(f'Total (train, test) pairs: {int(overview_df.n_pairs.sum())}')
display(overview_df)

Patients with model days: 21
Total (train, test) pairs: 558


,patient,n_model_days,n_test_days,n_pairs
0,YEY,2,2,1
1,YEZ,6,6,15
2,YFA,9,9,36
3,YFB,9,9,36
4,YFC,9,9,36
5,YFD,6,6,15
6,YFE,4,4,6
7,YFF,10,10,45
8,YFG,5,5,10
9,YFI,7,7,21


## 2. Phase-0 Status and Submission

One job per patient — fits global PCA bundle on all words from all days.

In [5]:
def p0_paths(patient):
    base_dir = VAD_ROOT / patient / 'functional_drift' / DRIFT_SUBDIR
    return dict(
        base_dir   = base_dir,
        bundle_pkl = base_dir / f'{patient}_global_bundle.pkl',
        meta_json  = base_dir / f'{patient}_global_bundle_meta.json',
        success    = base_dir / f'{patient}_GLOBAL_BUNDLE_SUCCESS',
        error      = base_dir / f'{patient}_global_bundle_error.txt',
    )


rows_p0 = []
for pt in all_patients:
    emb, cnt, dur = resolve_patient_inputs(pt)
    if any(p is None for p in [emb, cnt, dur]):
        continue
    if not get_test_days(pt):   # no source-run data at all
        continue
    pp   = p0_paths(pt)
    done = pp['success'].exists()
    err  = pp['error'].exists()
    rows_p0.append(dict(
        patient=pt,
        embeddings_path=emb, counts_path=cnt, durations_path=dur,
        **pp,
        done=done, has_error=err,
        needs_p0=not done and (FORCE_RESUBMIT or not done),
    ))

p0_df = pd.DataFrame(rows_p0)
print(f'Phase-0 rows: {len(p0_df)}')
print(f'  done:        {p0_df.done.sum()}')
print(f'  errors:      {p0_df.has_error.sum()}')
print(f'  needs submit:{p0_df.needs_p0.sum()}')
p0_df[['patient', 'done', 'has_error', 'needs_p0']]

Phase-0 rows: 21
  done:        21
  errors:      0
  needs submit:0


,patient,done,has_error,needs_p0
0,YEY,True,False,False
1,YEZ,True,False,False
2,YFA,True,False,False
3,YFB,True,False,False
4,YFC,True,False,False
5,YFD,True,False,False
6,YFE,True,False,False
7,YFF,True,False,False
8,YFG,True,False,False
9,YFI,True,False,False


In [6]:
DRY_RUN = False

submitted_p0 = {}   # patient -> job_id
failed_p0    = []

for _, row in p0_df[p0_df['needs_p0']].iterrows():
    pt      = row['patient']
    out_dir = row['base_dir']
    out_dir.mkdir(parents=True, exist_ok=True)

    if FORCE_RESUBMIT:
        reset = f"rm -f {row['success']} {row['error']}"
    else:
        reset = 'true'

    cmd = (
        f'{PYTHON_BIN} {WORKER_BUNDLE} {pt} {VAD_ROOT} {out_dir} '
        f'--source-run {SOURCE_RUN} '
        f'--n-pca {N_PCA} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--embeddings-path {row["embeddings_path"]} '
        f'--counts-path {row["counts_path"]} '
        f'--durations-path {row["durations_path"]}'
    )

    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=spca_bundle_{pt}',
        f'#SBATCH --partition={P0_PARTITION}',
        f'#SBATCH --cpus-per-task={P0_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        #f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P0_MEM}',
        f'#SBATCH --time={P0_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_p0_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_p0_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"',
        reset,
        cmd,
        'echo "END: $(date)"',
    ])

    sbatch_path = SCRIPTS_DIR / f'{pt}_p0.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] P0: {pt}')
        continue

    try:
        res = subprocess.run(['sbatch', '--parsable', str(sbatch_path)],
                             capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p0[pt] = jid
        print(f'submitted P0: {pt} -> {jid}')
    except subprocess.CalledProcessError as exc:
        failed_p0.append((pt, exc.stderr.strip()))
        print(f'FAILED P0: {pt}\n{exc.stderr}')

print(f'\nP0 submitted={len(submitted_p0)}, failed={len(failed_p0)}')


P0 submitted=0, failed=0


## 3. Phase-1 Status and Submission

One job per (patient, train_date) — fits full-day GLM with global bundle and fixed alpha.
Depends on Phase-0 bundle for the patient.

In [7]:
def p1_paths(patient, train_date):
    drift_dir = VAD_ROOT / patient / 'functional_drift' / DRIFT_SUBDIR / train_date
    return dict(
        out_dir   = drift_dir,
        model_tar = drift_dir / f'{patient}_fullday_model.tar',
        train_idx = drift_dir / f'{patient}_fullday_train_idx.npy',
        meta_json = drift_dir / f'{patient}_fullday_meta.json',
        success   = drift_dir / f'{patient}_TRAIN_SUCCESS',
        error     = drift_dir / f'{patient}_error.txt',
    )


rows_p1 = []
for pt in all_patients:
    emb, cnt, dur = resolve_patient_inputs(pt)
    if any(p is None for p in [emb, cnt, dur]):
        continue
    pp0 = p0_paths(pt)
    for train_date, cv_pkl, word_idx_path in get_model_days(pt):
        pp = p1_paths(pt, train_date)
        done  = pp['success'].exists()
        error = pp['error'].exists()
        rows_p1.append(dict(
            patient=pt, train_date=train_date,
            cv_results_pkl=cv_pkl, word_idx_path=word_idx_path,
            embeddings_path=emb, counts_path=cnt, durations_path=dur,
            global_bundle_pkl=pp0['bundle_pkl'],
            **pp,
            p0_done=pp0['success'].exists(),
            p0_jid=submitted_p0.get(pt, None),
            done=done, has_error=error,
            needs_p1=not done and (FORCE_RESUBMIT or not done),
        ))

p1_df = pd.DataFrame(rows_p1)
print(f'Phase-1 rows: {len(p1_df)}')
print(f'  done:        {p1_df.done.sum()}')
print(f'  errors:      {p1_df.has_error.sum()}')
print(f'  P0 done:     {p1_df.p0_done.sum()}')
print(f'  needs submit:{p1_df.needs_p1.sum()}')
p1_df[['patient','train_date','done','has_error','p0_done','needs_p1']]

Phase-1 rows: 156
  done:        156
  errors:      0
  P0 done:     156
  needs submit:0


,patient,train_date,done,has_error,p0_done,needs_p1
0,YEY,2024-04-01,True,False,True,False
1,YEY,2024-04-02,True,False,True,False
2,YEZ,2024-04-09,True,False,True,False
3,YEZ,2024-04-10,True,False,True,False
4,YEZ,2024-04-11,True,False,True,False
...,...,...,...,...,...,...
151,YFU,2025-12-18,True,False,True,False
152,YFV,2026-02-17,True,False,True,False
153,YFV,2026-02-18,True,False,True,False
154,YFV,2026-02-19,True,False,True,False


In [8]:
DRY_RUN = False

submitted_p1 = {}
failed_p1    = []

for _, row in p1_df[p1_df['needs_p1']].iterrows():
    pt         = row['patient']
    train_date = row['train_date']
    out_dir    = row['out_dir']
    out_dir.mkdir(parents=True, exist_ok=True)

    # Skip if no P0 bundle and no P0 job pending
    if not row['p0_done'] and not row['p0_jid']:
        print(f'  skipping {pt} {train_date}: P0 not done and no pending P0 job')
        continue

    if FORCE_RESUBMIT:
        reset = f"rm -f {row['success']} {row['error']}"
    else:
        reset = 'true'

    dep_flag = ''
    if row['p0_jid'] and not row['p0_done']:
        dep_flag = f'--dependency=afterok:{row["p0_jid"]}'

    gres_line = f'#SBATCH --gres={P1_GRES}' if P1_GRES else ''
    cmd = (
        f'{PYTHON_BIN} {WORKER_TRAIN} {pt} {VAD_ROOT} {out_dir} '
        f'--train-date {train_date} '
        f'--cv-results-pkl {row["cv_results_pkl"]} '
        f'--word-idx-path {row["word_idx_path"]} '
        f'--global-bundle-pkl {row["global_bundle_pkl"]} '
        f'--embeddings-path {row["embeddings_path"]} '
        f'--counts-path {row["counts_path"]} '
        f'--durations-path {row["durations_path"]} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER}'
    )

    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=spca_train_{pt}_{train_date}',
        f'#SBATCH --partition={P1_PARTITION}',
        gres_line,
        f'#SBATCH --cpus-per-task={P1_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        #f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P1_MEM}',
        f'#SBATCH --time={P1_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_{train_date}_p1_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_{train_date}_p1_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"',
        reset,
        cmd,
        'echo "END: $(date)"',
    ])

    sbatch_path = SCRIPTS_DIR / f'{pt}_{train_date}_p1.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] P1: {pt} {train_date}  dep={dep_flag}')
        continue

    try:
        sbatch_args = ['sbatch', '--parsable']
        if dep_flag:
            sbatch_args.append(dep_flag)
        sbatch_args.append(str(sbatch_path))
        res = subprocess.run(sbatch_args, capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p1[(pt, train_date)] = jid
        print(f'submitted P1: {pt} {train_date} -> {jid}  dep={dep_flag or "none"}')
    except subprocess.CalledProcessError as exc:
        failed_p1.append((pt, train_date, exc.stderr.strip()))
        print(f'FAILED P1: {pt} {train_date}\n{exc.stderr}')

print(f'\nP1 submitted={len(submitted_p1)}, failed={len(failed_p1)}')


P1 submitted=0, failed=0


## 4. Phase-2 Status and Submission

One job per (patient, train_date) — tests all subsequent days with permutation testing.
Depends on Phase-1 job. Uses same global bundle (no bundle re-fitting in perms).

In [9]:
def p2_success(patient, train_date, test_date, drift_dir):
    return drift_dir / f'{test_date}_DRIFT_SUCCESS'


rows_p2 = []
for pt in all_patients:
    test_days = get_test_days(pt)
    pp0 = p0_paths(pt)
    emb, cnt, dur = resolve_patient_inputs(pt)
    if any(p is None for p in [emb, cnt, dur]):
        continue
    for train_date, cv_pkl, word_idx_path in get_model_days(pt):
        pp = p1_paths(pt, train_date)
        future_tests = [t for t in test_days if t > train_date]
        if not future_tests:
            continue

        drift_dir     = pp['out_dir']
        all_test_done = all(
            p2_success(pt, train_date, td, drift_dir).exists()
            for td in future_tests
        )
        p1_pending = (pt, train_date) in submitted_p1

        rows_p2.append(dict(
            patient=pt, train_date=train_date,
            drift_dir=drift_dir,
            future_tests=future_tests,
            global_bundle_pkl=pp0['bundle_pkl'],
            embeddings_path=emb, counts_path=cnt, durations_path=dur,
            p1_done=pp['success'].exists(),
            p1_pending=p1_pending,
            all_test_done=all_test_done,
            p1_jid=submitted_p1.get((pt, train_date), None),
        ))

p2_df = pd.DataFrame(rows_p2)
print(f'Phase-2 rows: {len(p2_df)}')
print(f'  P1 done:        {p2_df.p1_done.sum()}')
print(f'  P1 pending:     {p2_df.p1_pending.sum()}')
print(f'  All tests done: {p2_df.all_test_done.sum()}')
print(f'  Can submit P2:  {(p2_df.p1_done | p2_df.p1_pending).sum()}')

Phase-2 rows: 135
  P1 done:        135
  P1 pending:     0
  All tests done: 135
  Can submit P2:  135


In [10]:
DRY_RUN = False

submitted_p2 = {}
failed_p2    = []

for _, row in p2_df.iterrows():
    pt         = row['patient']
    train_date = row['train_date']

    if not row['p1_done'] and not row['p1_pending']:
        continue
    if row['all_test_done'] and not FORCE_RESUBMIT:
        continue

    drift_dir = row['drift_dir']
    drift_dir.mkdir(parents=True, exist_ok=True)

    dep_flag = ''
    if row['p1_jid']:
        dep_flag = f'--dependency=afterok:{row["p1_jid"]}'
    elif not row['p1_done']:
        print(f'  skipping {pt} {train_date}: P1 not done and no pending job ID')
        continue

    gres_line = f'#SBATCH --gres={P2_GRES}' if P2_GRES else ''
    cmd = (
        f'{PYTHON_BIN} {WORKER_TEST} {pt} {VAD_ROOT} '
        f'--train-date {train_date} '
        f'--train-out-dir {drift_dir} '
        f'--global-bundle-pkl {row["global_bundle_pkl"]} '
        f'--source-run {SOURCE_RUN} '
        f'--n-permutations {N_PERMUTATIONS} '
        f'--n-jobs {N_JOBS_PERM} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--embeddings-path {row["embeddings_path"]} '
        f'--counts-path {row["counts_path"]} '
        f'--durations-path {row["durations_path"]}'
    )

    script = '\n'.join([
        '#!/bin/bash',
        f'#SBATCH --job-name=spca_test_{pt}_{train_date}',
        f'#SBATCH --partition={P2_PARTITION}',
        gres_line,
        f'#SBATCH --cpus-per-task={P2_CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        #f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={P2_MEM}',
        f'#SBATCH --time={P2_TIME}',
        f'#SBATCH --output={LOGS_DIR}/{pt}_{train_date}_p2_%j.out',
        f'#SBATCH --error={LOGS_DIR}/{pt}_{train_date}_p2_%j.err',
        '',
        'set -eo pipefail',
        CONDA_ACTIVATE,
        'echo "HOST: $(hostname)  START: $(date)"',
        cmd,
        'echo "END: $(date)"',
    ])

    sbatch_path = SCRIPTS_DIR / f'{pt}_{train_date}_p2.sbatch'
    sbatch_path.write_text(script)

    if DRY_RUN:
        print(f'[DRY] P2: {pt} {train_date}  dep={dep_flag}')
        continue

    try:
        sbatch_args = ['sbatch', '--parsable']
        if dep_flag:
            sbatch_args.append(dep_flag)
        sbatch_args.append(str(sbatch_path))
        res = subprocess.run(sbatch_args, capture_output=True, text=True, check=True)
        jid = res.stdout.strip()
        submitted_p2[(pt, train_date)] = jid
        print(f'submitted P2: {pt} {train_date} -> {jid}  dep={dep_flag or "none"}')
    except subprocess.CalledProcessError as exc:
        failed_p2.append((pt, train_date, exc.stderr.strip()))
        print(f'FAILED P2: {pt} {train_date}\n{exc.stderr}')

print(f'\nP2 submitted={len(submitted_p2)}, failed={len(failed_p2)}')


P2 submitted=0, failed=0


## 5. Status Check

In [ ]:
# Run this cell at any time to check current status without submitting.
status_rows = []
for pt in all_patients:
    pp0      = p0_paths(pt)
    mdays    = get_model_days(pt)
    tdays    = get_test_days(pt)

    p0_done  = pp0['success'].exists()
    p0_err   = pp0['error'].exists()

    for train_date, _, _ in mdays:
        pp    = p1_paths(pt, train_date)
        p1_ok = pp['success'].exists()
        p1_err = pp['error'].exists()

        future = [t for t in tdays if t > train_date]
        n_p2_done = sum(
            1 for td in future
            if (pp['out_dir'] / f'{td}_DRIFT_SUCCESS').exists()
        )
        n_p2_err = sum(
            1 for td in future
            if (pp['out_dir'] / f'{td}_error.txt').exists()
        )
        status_rows.append(dict(
            patient=pt, train_date=train_date,
            p0_done=p0_done, p0_err=p0_err,
            p1_done=p1_ok, p1_err=p1_err,
            n_future_days=len(future),
            n_p2_done=n_p2_done, n_p2_err=n_p2_err,
        ))

status_df = pd.DataFrame(status_rows)
if not status_df.empty:
    print(f'P0 done: {status_df.p0_done.sum()} / {status_df.patient.nunique()} patients')
    print(f'P1 done: {status_df.p1_done.sum()} / {len(status_df)} train-days')
    total_p2 = status_df.n_future_days.sum()
    done_p2  = status_df.n_p2_done.sum()
    err_p2   = status_df.n_p2_err.sum()
    print(f'P2 done: {done_p2} / {total_p2} test-pairs   errors: {err_p2}')
    display(status_df)
else:
    print('No data found.')

P0 done: 156 / 21 patients
P1 done: 156 / 156 train-days
P2 done: 558 / 558 test-pairs   errors: 0


,patient,train_date,p0_done,p0_err,p1_done,p1_err,n_future_days,n_p2_done,n_p2_err
0,YEY,2024-04-01,True,False,True,False,1,1,0
1,YEY,2024-04-02,True,False,True,False,0,0,0
2,YEZ,2024-04-09,True,False,True,False,5,5,0
3,YEZ,2024-04-10,True,False,True,False,4,4,0
4,YEZ,2024-04-11,True,False,True,False,3,3,0
...,...,...,...,...,...,...,...,...,...
151,YFU,2025-12-18,True,False,True,False,0,0,0
152,YFV,2026-02-17,True,False,True,False,3,3,0
153,YFV,2026-02-18,True,False,True,False,2,2,0
154,YFV,2026-02-19,True,False,True,False,1,1,0
